# Workflow: English to SQL

This notebook builds a small **workflow** — a fixed sequence of LLM calls and deterministic steps chained together — that lets you ask a database questions in plain English.

### The Pipeline
1. **Text → SQL:** The model translates a natural-language question into a SQL query, grounded in the database schema.
2. **Execute:** The query runs against a real (in-memory) SQLite database.
3. **Result → Text:** The model translates the raw query results back into a friendly, direct answer.

Unlike an agent, there's no looping or tool-choice — each step always runs in the same order.

---
## Step 1: Install Dependencies
`sqlite3` ships with Python, so we only need the OpenAI SDK.

In [ ]:
!pip install -q openai python-dotenv

---
## Step 2: Set Up Environment Variables & Client

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

client = OpenAI()

---
## Step 3: Set Up a Sample Database
A tiny in-memory SQLite database with one `employees` table stands in for a real production database.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        salary INTEGER,
        hire_date TEXT
    )
""")

employees_data = [
    (1, "Alice", "Engineering", 120000, "2021-03-15"),
    (2, "Bob", "Engineering", 110000, "2022-06-01"),
    (3, "Charlie", "Marketing", 85000, "2020-01-10"),
    (4, "Diana", "Sales", 95000, "2019-11-23"),
    (5, "Evan", "Marketing", 90000, "2023-02-18"),
]
cursor.executemany("INSERT INTO employees VALUES (?,?,?,?,?)", employees_data)
conn.commit()

SCHEMA = """
Table: employees
Columns: id (INTEGER), name (TEXT), department (TEXT), salary (INTEGER), hire_date (TEXT)
"""

---
## Step 4: Text → SQL
The model only sees the schema, not the data — it generates a query, it doesn't guess values.

In [ ]:
def text_to_sql(user_prompt: str) -> str:
    system_instruction = f"""
    You are a SQL generator for SQLite.
    Given the database schema:
    {SCHEMA}
    Generate ONLY a valid SQL query to answer the user's request. 
    Do NOT wrap the output in markdown code blocks, backticks, or extra commentary.
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()

---
## Step 5: Execute the SQL
A plain SQLite call — no LLM involved. Errors are caught and returned rather than raised, so the pipeline can report them gracefully instead of crashing.

In [ ]:
def run_sql(query: str):
    try:
        cursor.execute(query)
        columns = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()
        return {"columns": columns, "rows": rows, "error": None}
    except Exception as e:
        return {"columns": [], "rows": [], "error": str(e)}

---
## Step 6: Result → Text
The raw SQL result (columns + rows) is handed back to the model along with the original question, so it can phrase a direct, human-friendly answer.

In [ ]:
def sql_result_to_text(user_prompt: str, sql_query: str, db_result: dict) -> str:
    system_instruction = (
        "You are an assistant providing friendly, accurate answers based on database query results. "
        "Answer the user's question directly and concisely."
    )

    context = f"""
    User Question: {user_prompt}
    SQL Executed: {sql_query}
    Query Results: {db_result}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": context},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

---
## Step 7: Chain It Into a Full Pipeline

In [ ]:
def ask_database(user_question: str):
    print(f"\n🗣️ User Input: '{user_question}'")

    # Pipeline Step 1
    sql = text_to_sql(user_question)
    print(f"⚙️ Generated SQL: {sql}")

    # Pipeline Step 2
    result = run_sql(sql)
    if result["error"]:
        print(f"❌ SQL Execution Error: {result['error']}")
        return

    # Pipeline Step 3
    final_answer = sql_result_to_text(user_question, sql, result)
    print(f"💬 Final Answer: {final_answer}")

---
## Step 8: Run Example Queries

In [ ]:
ask_database("Who is the highest paid person in Engineering?")
ask_database("What is the average salary across all departments?")